# 01 - Advanced Trust Audit

## Purpose

Create a structured trust audit table for the Night Shift extension.

This notebook turns validation checks into a reusable Gold audit output that can show whether key project tables are trustworthy enough for reporting.

## Expected output

`vattenfall_dev.analytics.gold_data_trust_audit`

## Main checks

- null key dimensions
- duplicate grains
- missing joins
- invalid values
- table readiness

## Main idea

Instead of only printing validation results, this notebook stores trust evidence as a structured Gold table.

In [0]:
import sys
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql import functions as F

repo_src_path = "/Workspace/Users/dirella@startsteps.org/vattenfall-week9-capstone-anna/src"

if repo_src_path not in sys.path:
    sys.path.append(repo_src_path)

from validation.night_shift_checks import count_duplicate_grain, count_nulls

In [0]:
catalog = "vattenfall_dev"

tables = {
    "silver_market_prices": f"{catalog}.refined.silver_market_prices",
    "silver_weather": f"{catalog}.refined.silver_weather",
    "silver_grid_events": f"{catalog}.refined.silver_grid_events",
    "silver_asset_reference": f"{catalog}.refined.silver_asset_reference",
    "gold_regional_operations_dashboard": f"{catalog}.analytics.gold_regional_operations_dashboard",
}

target_table = f"{catalog}.analytics.gold_data_trust_audit"

print("Night Shift output:", target_table)

In [0]:
audit_rows = []
check_timestamp = datetime.utcnow().isoformat()

def add_audit_row(check_name, source_table, failed_record_count, severity):
    status = "PASS" if failed_record_count == 0 else "FAIL"
    audit_rows.append(
        Row(
            check_name=check_name,
            source_table=source_table,
            check_status=status,
            failed_record_count=int(failed_record_count),
            severity=severity,
            check_timestamp=check_timestamp,
        )
    )

In [0]:
null_checks = {
    tables["silver_market_prices"]: ["report_day", "region"],
    tables["silver_weather"]: ["report_day", "region"],
    tables["silver_grid_events"]: ["event_day", "region", "asset_id"],
    tables["silver_asset_reference"]: ["asset_id", "region"],
    tables["gold_regional_operations_dashboard"]: ["report_day", "region"],
}

for table_name, columns in null_checks.items():
    df = spark.table(table_name)

    for column_name in columns:
        failed_count = count_nulls(df, column_name)

        add_audit_row(
            check_name=f"null_check_{column_name}",
            source_table=table_name,
            failed_record_count=failed_count,
            severity="HIGH",
        )

In [0]:
duplicate_checks = {
    tables["silver_asset_reference"]: ["asset_id"],
    tables["gold_regional_operations_dashboard"]: ["report_day", "region"],
}

for table_name, grain_columns in duplicate_checks.items():
    df = spark.table(table_name)

    failed_count = count_duplicate_grain(df, grain_columns)

    add_audit_row(
        check_name=f"duplicate_grain_{'_'.join(grain_columns)}",
        source_table=table_name,
        failed_record_count=failed_count,
        severity="HIGH",
    )

In [0]:
market_df = spark.table(tables["silver_market_prices"])
weather_df = spark.table(tables["silver_weather"])
grid_df = spark.table(tables["silver_grid_events"])

invalid_checks = [
    (
        "negative_market_volume",
        tables["silver_market_prices"],
        market_df.filter(F.col("volume_mwh") < 0).count(),
        "MEDIUM",
    ),
    (
        "negative_wind_speed",
        tables["silver_weather"],
        weather_df.filter(F.col("wind_speed_kmh") < 0).count(),
        "MEDIUM",
    ),
    (
        "negative_grid_duration",
        tables["silver_grid_events"],
        grid_df.filter(F.col("duration_minutes") < 0).count(),
        "HIGH",
    ),
]

for check_name, source_table, failed_count, severity in invalid_checks:
    add_audit_row(
        check_name=check_name,
        source_table=source_table,
        failed_record_count=failed_count,
        severity=severity,
    )

In [0]:
grid_df = spark.table(tables["silver_grid_events"])
asset_df = spark.table(tables["silver_asset_reference"])

missing_asset_matches = (
    grid_df.alias("g")
    .join(asset_df.alias("a"), on="asset_id", how="left")
    .filter(F.col("a.asset_name").isNull())
    .count()
)

add_audit_row(
    check_name="missing_asset_reference_match",
    source_table=tables["silver_grid_events"],
    failed_record_count=missing_asset_matches,
    severity="HIGH",
)

In [0]:
audit_df = spark.createDataFrame(audit_rows)

display(audit_df)

In [0]:
(
    audit_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print("Written Night Shift trust audit table:", target_table)
print("Audit rows:", audit_df.count())

In [0]:
result_df = spark.table(target_table)

print("Rows in trust audit table:", result_df.count())

display(result_df)

print("Trust check status distribution:")
result_df.groupBy("check_status").count().show()